# **Tahap 4 - Preprocessing untuk Machine Learning**

## Import Library

In [15]:
import numpy as np
import pandas as pd
from pathlib import Path

## Path Konfigurasi

In [16]:
BASE_DIR     = Path(__file__).resolve().parent.parent \
               if "__file__" in dir() else Path.cwd().parent
 
CLEANED_DIR  = BASE_DIR / "data" / "cleaned"
FEATURES_DIR = BASE_DIR / "data" / "features"
 
if not CLEANED_DIR.exists():
    raise FileNotFoundError(
        f"Folder data/cleaned tidak ditemukan: {CLEANED_DIR}\n"
        f"Jalankan dulu: python src/02_cleaning.py"
    )
 
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

## Konstanta

In [17]:
TRAIN_YEARS = list(range(2015, 2022))   # 2015–2021
TEST_YEARS  = list(range(2022, 2025))   # 2022–2024
 
PROVINCE_GEO = {
    "Aceh"                     : ("Sumatera",    "Sumatera Bagian Utara"),
    "Sumatera Utara"           : ("Sumatera",    "Sumatera Bagian Utara"),
    "Sumatera Barat"           : ("Sumatera",    "Sumatera Bagian Tengah"),
    "Riau"                     : ("Sumatera",    "Sumatera Bagian Tengah"),
    "Jambi"                    : ("Sumatera",    "Sumatera Bagian Tengah"),
    "Sumatera Selatan"         : ("Sumatera",    "Sumatera Bagian Selatan"),
    "Bengkulu"                 : ("Sumatera",    "Sumatera Bagian Selatan"),
    "Lampung"                  : ("Sumatera",    "Sumatera Bagian Selatan"),
    "Kepulauan Bangka Belitung": ("Sumatera",    "Sumatera Bagian Selatan"),
    "Kepulauan Riau"           : ("Sumatera",    "Sumatera Bagian Tengah"),
    "DKI Jakarta"              : ("Jawa",        "Jawa Bagian Barat"),
    "Jawa Barat"               : ("Jawa",        "Jawa Bagian Barat"),
    "Banten"                   : ("Jawa",        "Jawa Bagian Barat"),
    "Jawa Tengah"              : ("Jawa",        "Jawa Bagian Tengah"),
    "DI Yogyakarta"            : ("Jawa",        "Jawa Bagian Tengah"),
    "Jawa Timur"               : ("Jawa",        "Jawa Bagian Timur"),
    "Bali"                     : ("Bali Nusra",  "Bali"),
    "Nusa Tenggara Barat"      : ("Bali Nusra",  "Nusa Tenggara"),
    "Nusa Tenggara Timur"      : ("Bali Nusra",  "Nusa Tenggara"),
    "Kalimantan Barat"         : ("Kalimantan",  "Kalimantan Bagian Barat"),
    "Kalimantan Tengah"        : ("Kalimantan",  "Kalimantan Bagian Tengah"),
    "Kalimantan Selatan"       : ("Kalimantan",  "Kalimantan Bagian Selatan"),
    "Kalimantan Timur"         : ("Kalimantan",  "Kalimantan Bagian Timur"),
    "Kalimantan Utara"         : ("Kalimantan",  "Kalimantan Bagian Timur"),
    "Sulawesi Utara"           : ("Sulawesi",    "Sulawesi Bagian Utara"),
    "Gorontalo"                : ("Sulawesi",    "Sulawesi Bagian Utara"),
    "Sulawesi Tengah"          : ("Sulawesi",    "Sulawesi Bagian Tengah"),
    "Sulawesi Barat"           : ("Sulawesi",    "Sulawesi Bagian Barat"),
    "Sulawesi Selatan"         : ("Sulawesi",    "Sulawesi Bagian Selatan"),
    "Sulawesi Tenggara"        : ("Sulawesi",    "Sulawesi Bagian Selatan"),
    "Maluku"                   : ("Maluku Papua","Maluku"),
    "Maluku Utara"             : ("Maluku Papua","Maluku"),
    "Papua Barat"              : ("Maluku Papua","Papua"),
    "Papua"                    : ("Maluku Papua","Papua"),
}

## Step 1 - Load dan Tambah Metadata

In [18]:
def add_province_metadata(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['pulau']  = df['provinsi'].map(lambda p: PROVINCE_GEO.get(p, (None, None))[0])
    df['region'] = df['provinsi'].map(lambda p: PROVINCE_GEO.get(p, (None, None))[1])
 
    missing_geo = df[df['pulau'].isna()]['provinsi'].unique()
    if len(missing_geo):
        print(f"  ⚠️  Provinsi tanpa metadata geo: {list(missing_geo)}")
 
    return df

## Step 2 - Imputasi Missing Values

In [19]:
def impute_missing(df: pd.DataFrame) -> pd.DataFrame:
    """
    Strategi imputasi:
      - crime_rate_per100k  → median per pulau per tahun
      - population          → median per pulau per tahun
      - population_density  → median per pulau per tahun
      - unemployment_rate   → median per pulau (hanya ada untuk 2024+)
    """
    df = df.copy()
    target_cols = ['crime_rate_per100k', 'population', 'population_density']
 
    print(f"\n  Missing values sebelum imputasi:")
    for col in target_cols + ['unemployment_rate']:
        if col in df.columns:
            n = df[col].isna().sum()
            pct = n / len(df) * 100
            print(f"    {col:<25}: {n:>3} ({pct:.1f}%)")

    for col in target_cols:
        if col not in df.columns:
            continue
        group_median = df.groupby(['pulau', 'tahun'])[col].transform('median')
        fallback     = df.groupby('tahun')[col].transform('median')
        df[col] = df[col].fillna(group_median).fillna(fallback)

    if 'unemployment_rate' in df.columns:
        group_median_u = df.groupby('pulau')['unemployment_rate'].transform('median')
        df['unemployment_rate'] = df['unemployment_rate'].fillna(group_median_u)
 
    print(f"\n  Missing values sesudah imputasi:")
    for col in target_cols + ['unemployment_rate']:
        if col in df.columns:
            n = df[col].isna().sum()
            pct = n / len(df) * 100
            flag = "✅" if n == 0 else "⚠️ "
            print(f"    {flag} {col:<25}: {n:>3} ({pct:.1f}%)")
 
    return df

## Step 03 - Label Encoding

In [20]:
def label_encode(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    df = df.copy()
    encoding_map = {}
 
    for col in ['pulau', 'region']:
        if col not in df.columns:
            continue
        categories = sorted(df[col].dropna().unique())
        mapping    = {cat: idx for idx, cat in enumerate(categories)}
        df[f'{col}_enc'] = df[col].map(mapping)
        encoding_map[col] = mapping
        print(f"  ✅ {col:<8} → {col}_enc  | {mapping}")
 
    return df, encoding_map


## Step 04 - Normalisasi (StandardScaler)

In [21]:
NUMERIC_COLS = [
    'crime_rate_per100k',
    'population',
    'population_density',
    'unemployment_rate',
] 
 
def standard_scale(df: pd.DataFrame, fit_mask: pd.Series) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    StandardScaler manual: z = (x - mean) / std
    Fit HANYA dari data training (fit_mask), apply ke semua.
 
    Return:
      df_scaled   : DataFrame dengan kolom _scaled ditambahkan
      scaler_info : DataFrame berisi mean & std tiap kolom (untuk disimpan)
    """
    df       = df.copy()
    df_train = df[fit_mask]
    scaler_records = []
 
    for col in NUMERIC_COLS:
        if col not in df.columns:
            continue
        mean = df_train[col].mean()
        std  = df_train[col].std()
        if std == 0 or pd.isna(std):
            std = 1.0   
 
        df[f'{col}_scaled'] = (df[col] - mean) / std
        scaler_records.append({'feature': col, 'mean': round(mean, 4), 'std': round(std, 4)})
        print(f"  ✅ {col:<25} → mean={mean:.2f}, std={std:.2f}")
 
    scaler_info = pd.DataFrame(scaler_records)
    return df, scaler_info

## Step 05 - Train Test Split

In [22]:
def time_split(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    train = df[df['tahun'].isin(TRAIN_YEARS)].copy()
    test  = df[df['tahun'].isin(TEST_YEARS)].copy()
    return train, test

## Main Program

In [23]:
def main():
    print("\n" + "=" * 60)
    print("TAHAP 4 — PREPROCESSING UNTUK ML")
    print(f"Input  : {CLEANED_DIR}")
    print(f"Output : {FEATURES_DIR}")
    print("=" * 60)
 
    df = pd.read_csv(CLEANED_DIR / "crime_merged.csv")
    df['tahun'] = df['tahun'].astype(int)
    print(f"\nLoaded crime_merged.csv : {df.shape[0]} rows × {df.shape[1]} kolom")
 
    print("\n--- Step 1: Tambah Metadata Provinsi ---")
    df = add_province_metadata(df)
    print(f"  ✅ Kolom ditambah: pulau, region")

    print("\n--- Step 2: Imputasi Missing Values ---")
    df = impute_missing(df)

    print("\n--- Step 3: Label Encoding ---")
    df, encoding_map = label_encode(df)

    print("\n--- Step 4: Normalisasi (StandardScaler) ---")
    train_mask       = df['tahun'].isin(TRAIN_YEARS)
    df, scaler_info  = standard_scale(df, train_mask)

    print("\n--- Step 5: Train/Test Split ---")
    train_df, test_df = time_split(df)
    print(f"  ✅ Train : {len(train_df)} rows | tahun {TRAIN_YEARS[0]}–{TRAIN_YEARS[-1]}")
    print(f"  ✅ Test  : {len(test_df)} rows  | tahun {TEST_YEARS[0]}–{TEST_YEARS[-1]}")

    print("\n--- Simpan Output ---")

    df.to_csv(FEATURES_DIR / "ml_dataset.csv", index=False)
    print(f"  ✅ ml_dataset.csv       → {len(df)} rows × {len(df.columns)} kolom")

    scaled_cols  = ['provinsi', 'tahun', 'pulau', 'region', 'pulau_enc', 'region_enc']
    scaled_cols += [c for c in df.columns if c.endswith('_scaled')]
    df[scaled_cols].to_csv(FEATURES_DIR / "ml_dataset_scaled.csv", index=False)
    print(f"  ✅ ml_dataset_scaled.csv→ {len(scaled_cols)} kolom")

    scaler_info.to_csv(FEATURES_DIR / "scaler_info.csv", index=False)
    print(f"  ✅ scaler_info.csv      → parameter mean & std")

    train_df.to_csv(FEATURES_DIR / "train.csv", index=False)
    test_df.to_csv( FEATURES_DIR / "test.csv",  index=False)
    print(f"  ✅ train.csv            → {len(train_df)} rows")
    print(f"  ✅ test.csv             → {len(test_df)} rows")

    print("\n" + "=" * 60)
    print("SUMMARY KOLOM ml_dataset.csv")
    print("=" * 60)
    col_groups = {
        "Identitas"   : ['provinsi', 'tahun', 'pulau', 'region'],
        "Raw numerik" : ['crime_total', 'crime_rate_per100k', 'population',
                         'population_density', 'unemployment_rate'],
        "Encoded"     : ['pulau_enc', 'region_enc'],
        "Scaled"      : [c for c in df.columns if c.endswith('_scaled')],
        "Flag"        : ['is_outlier', 'pop_extrapolated'],
    }
    for group, cols in col_groups.items():
        available = [c for c in cols if c in df.columns]
        print(f"  {group:<14}: {available}")
 
 
if __name__ == "__main__":
    main()


TAHAP 4 — PREPROCESSING UNTUK ML
Input  : /Users/paulinadevinawijaya/Desktop/CRIME IN INDONESIA PORT/data/cleaned
Output : /Users/paulinadevinawijaya/Desktop/CRIME IN INDONESIA PORT/data/features

Loaded crime_merged.csv : 331 rows × 9 kolom

--- Step 1: Tambah Metadata Provinsi ---
  ✅ Kolom ditambah: pulau, region

--- Step 2: Imputasi Missing Values ---

  Missing values sebelum imputasi:
    crime_rate_per100k       :  24 (7.3%)
    population               :  23 (6.9%)
    population_density       :  23 (6.9%)
    unemployment_rate        : 298 (90.0%)

  Missing values sesudah imputasi:
    ✅ crime_rate_per100k       :   0 (0.0%)
    ✅ population               :   0 (0.0%)
    ✅ population_density       :   0 (0.0%)
    ✅ unemployment_rate        :   0 (0.0%)

--- Step 3: Label Encoding ---
  ✅ pulau    → pulau_enc  | {'Bali Nusra': 0, 'Jawa': 1, 'Kalimantan': 2, 'Maluku Papua': 3, 'Sulawesi': 4, 'Sumatera': 5}
  ✅ region   → region_enc  | {'Bali': 0, 'Jawa Bagian Barat': 1, 'Ja